# Graph Inspection Notebook

This notebook loads and inspects the knowledge graph using the utility functions from `cli/commands/evals/utils.py`.

In [16]:
from pathlib import Path
import sys
import polars as pl

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from cli.commands.evals.utils import (
    load_graph,
    load_node_metadata,
    compute_pagerank,
    pagerank_to_dataframe,
)

## Load Graph

In [17]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
edges_path = Path("data/gold/kg/parquet/edges.parquet")

G, edge_type_lookup, node_types, edge_types = load_graph(nodes_path, edges_path)

[03/16/26 23:31:38] INFO     Loaded 192121 nodes with 10 node types: ['ANA', 'BPO', 'CCO', 'DIS',       ]8;id=385496;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=55480;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#42\42]8;;\
                             'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']                                             

[03/16/26 23:32:40] INFO     Loaded 21851915 edges with 26 edge types: ['ANA-ANA', 'ANA-PRO',           ]8;id=828370;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=120791;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#61\61]8;;\
                             'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE',                     
                             'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO',                     
                             'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN',                     
                             'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']                     

## Basic Statistics

In [18]:
print(f"Nodes: {G.number_of_nodes():,}")
print(f"Edges: {G.number_of_edges():,}")
print(f"\nNode types ({len(node_types)}): {node_types}")
print(f"\nEdge types ({len(edge_types)}): {edge_types}")

Nodes: 192,123
Edges: 21,851,915

Node types (10): ['ANA', 'BPO', 'CCO', 'DIS', 'DRG', 'EXP', 'GEN', 'MFN', 'PHE', 'PWY']

Edge types (26): ['ANA-ANA', 'ANA-PRO', 'BPO-BPO', 'BPO-PRO', 'CCO-CCO', 'CCO-PRO', 'DIS-DIS', 'DIS-PHE', 'DIS-PRO', 'DRG-DIS', 'DRG-DRG', 'DRG-PHE', 'DRG-PRO', 'EXP-BPO', 'EXP-CCO', 'EXP-DIS', 'EXP-EXP', 'EXP-MFN', 'EXP-PRO', 'MFN-MFN', 'MFN-PRO', 'PHE-PHE', 'PHE-PRO', 'PRO-PRO', 'PWY-PRO', 'PWY-PWY']


## Load Node Metadata

In [19]:
node_metadata = load_node_metadata(nodes_path)
node_metadata.head(10)

[03/16/26 23:32:53] INFO     Loaded metadata for 192121 nodes                                           ]8;id=298772;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=29611;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#98\98]8;;\

id,label,name
str,str,str
"""ENSG00000000003""","""GEN""","""tetraspanin 6"""
"""ENSG00000000005""","""GEN""","""tenomodulin"""
"""ENSG00000000419""","""GEN""","""dolichyl-phosphate mannosyltra…"
"""ENSG00000000457""","""GEN""","""SCY1 like pseudokinase 3"""
"""ENSG00000000460""","""GEN""","""FIGNL1 interacting regulator o…"
"""ENSG00000000938""","""GEN""","""FGR proto-oncogene, Src family…"
"""ENSG00000000971""","""GEN""","""complement factor H"""
"""ENSG00000001036""","""GEN""","""alpha-L-fucosidase 2"""
"""ENSG00000001084""","""GEN""","""glutamate-cysteine ligase cata…"


In [20]:
# Node counts by type
node_metadata.group_by("label").len().sort("len", descending=True)

# Node counts by type from 

label,len
str,u32
"""GEN""",61306
"""DIS""",36431
"""BPO""",25754
"""PHE""",19341
"""DRG""",16766
"""ANA""",14624
"""MFN""",10161
"""CCO""",4052
"""PWY""",2805


In [31]:
# Find node where id == GO_0009199 or GO_0009161
selected_ids = ["GO_0009199", "GO_0009161"]
selected_nodes = node_metadata.filter(pl.col("id").is_in(selected_ids))

In [33]:
# Compute pagerank scores from graph
pagerank_scores = compute_pagerank(G)
# Convert to dataframe
pagerank_df = pagerank_to_dataframe(pagerank_scores, node_metadata)
# Filter for selected nodes
pagerank_df.filter(pl.col("id").is_in(selected_ids))

[03/16/26 23:35:55] INFO     Computing PageRank with alpha=0.85...                                     ]8;id=652900;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=405462;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#112\112]8;;\

[03/16/26 23:36:15] INFO     Computed PageRank for 192123 nodes                                        ]8;id=466225;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py\utils.py]8;;\:]8;id=182334;file:///Users/ayush/Documents/Zitnik_Lab/optimus/cli/commands/evals/utils.py#114\114]8;;\

id,pagerank,label,name
str,f64,str,str
"""GO_0009199""",0.000002,null,null
"""GO_0009161""",0.000002,null,null


In [ ]:
selected_ids = ["GO_0009199", "GO_0009161"]
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
df = pl.read_parquet(nodes_path)
df.filter(pl.col("id").is_in(selected_ids))

id,label,properties
str,str,str


In [43]:
edges_path = Path("data/gold/kg/parquet/edges.parquet")
edges_df = pl.read_parquet(edges_path)
edges_df.filter(pl.col("from").is_in(selected_ids) | pl.col("to").is_in(selected_ids))

from,to,label,relation,undirected,properties
str,str,str,str,bool,str
"""GO_0009199""","""ENSG00000170525""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0014783194…"
"""GO_0009199""","""ENSG00000232810""","""DIS-PRO""","""ASSOCIATED_WITH""",true,"""{""evidence_score"":0.0022174791…"
"""GO_0009123""","""GO_0009161""","""DIS-DIS""","""PARENT""",false,"""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0009141""","""GO_0009199""","""DIS-DIS""","""PARENT""",false,"""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0009161""","""GO_0009156""","""DIS-DIS""","""PARENT""",false,"""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0009161""","""GO_0009167""","""DIS-DIS""","""PARENT""",false,"""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0009199""","""GO_0009201""","""DIS-DIS""","""PARENT""",false,"""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0009199""","""GO_0009205""","""DIS-DIS""","""PARENT""",false,"""{""sources"":{""direct"":[""OPEN_TA…"


In [32]:
# Read in pagerank df
pagerank_df = pl.read_csv("data/gold/evals/pagerank.csv")

# Filter for selected nodes
pagerank_df.filter(pl.col("id").is_in(selected_ids))

rank,id,label,name,pagerank
i64,str,str,str,f64
103889,"""GO_0009199""",null,null,0.000002
117515,"""GO_0009161""",null,null,0.000002


## Debug Difference

In [13]:
nodes_path = Path("data/gold/kg/parquet/nodes.parquet")
df = pl.read_parquet(nodes_path)

print("Rows:", df.height)
print("Unique IDs:", df.select(pl.col("id").n_unique()).item())
print("Duplicate rows by ID:", df.height - df.select(pl.col("id").n_unique()).item())

Rows: 192682
Unique IDs: 192123
Duplicate rows by ID: 559


In [15]:
# Identify IDs that appear more than once
duplicate_ids = (
    df.group_by("id")
    .len()
    .filter(pl.col("len") > 1)
    .select("id")
)

# Join back to original data to get full metadata for those IDs
duplicated_nodes_table = (
    df.join(duplicate_ids, on="id", how="inner")
    .with_columns(
        pl.col("properties").str.json_path_match("$.name").alias("name")
    )
    .select(["id", "name", "label", "properties"])
    .sort("id")
)

# Save the table
output_file = Path("data/gold/evals/duplicated_nodes.csv")
duplicated_nodes_table.write_csv(output_file)

print(f"Found {duplicate_ids.height} unique duplicated IDs resulting in {duplicated_nodes_table.height} total rows.")
print(f"Table saved to: {output_file}")

# Display the first few rows for inspection
duplicated_nodes_table.head(10)

Found 559 unique duplicated IDs resulting in 1118 total rows.
Table saved to: data/gold/evals/duplicated_nodes.csv


id,name,label,properties
str,str,str,str
"""GO_0000002""","""mitochondrial genome maintenan…","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000002""","""obsolete mitochondrial genome …","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000050""","""urea cycle""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000050""","""urea cycle""","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000096""","""sulfur amino acid metabolic pr…","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000096""","""sulfur amino acid metabolic pr…","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000226""","""microtubule cytoskeleton organ…","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
"""GO_0000226""","""microtubule cytoskeleton organ…","""BPO""","""{""sources"":{""direct"":[""GO""],""i…"
"""GO_0000278""","""mitotic cell cycle""","""DIS""","""{""sources"":{""direct"":[""OPEN_TA…"
